In [ ]:
from IPython.core.display import HTML
with open ("../style.css") as file:
    css = file.read()
HTML(css)

# A Grammar for Propositional Logic

This file shows how a simple symbolic calculator can be implemented using `Ply`.  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{formula}     & \rightarrow & \texttt{formula}\; \texttt{'∨'} \; \texttt{formula}            \\
                       & \mid        & \texttt{conjunction}                                           \\[0.2cm]
  \texttt{conjunction} & \rightarrow & \texttt{conjunction}\; \texttt{'∧'} \; \texttt{negation}       \\
                       & \mid        & \texttt{negation}                                              \\[0.2cm]
  \texttt{negation}    & \rightarrow & \texttt{'¬'}\; \texttt{negation}                               \\
                       & \mid        & \;\texttt{'('} \; \texttt{formula} \;\texttt{')'}              \\
                       & \mid        & \;\texttt{ID}                        
\end{array}
$$

## Specification of the Scanner

In [ ]:
import ply.lex as lex

There are only five tokens that need to be defined via regular expressions.  The other tokens consist only of a single character and are therefore 
defined as literals.

In [ ]:
tokens = [ 'ID' ]

The token `ID` specifies the name of a propositional *variable*.  It may contain the angle brackets `<` and `>`.

In [ ]:
t_ID = r'[a-zA-Z]+'

`literals` is a list operator symbols that consist of a single character.

Alternativ:
```
literals = "∧∨¬()"
```

In [ ]:
literals = ['∧', '∨', '¬', '(', ')']

Blanks and tabulators are ignored.

In [ ]:
t_ignore  = ' \t\n'

Unkown characters are reported as lexical errors.

In [ ]:
def t_error(_):
    pass

In [ ]:
__file__ = 'main'

We generate the lexer.

In [ ]:
lexer = lex.lex()

## Specification of the Parser

In [ ]:
from ply import yacc

We use the following abbreviations when implementing the grammar:
  * `f := formula`
  * `c := conjunction`
  * `n := negation`

In [ ]:
def p_f1(p):
    "f : f '∨' c"
    p[0] = ('∨', p[1], p[3])
    
def p_f2(p):
    "f : c"
    p[0] = p[1]
    
def p_c1(p):
    "c : c '∧' n"
    p[0] = ('∧', p[1], p[3])
    
def p_c2(p):
    "c : n"
    p[0] = p[1]
    
def p_n1(p):
    "n : '¬' n"
    p[0] = ('¬', p[2])
    
def p_n2(p):
    "n : '(' f ')'"
    p[0] = p[2]
    
def p_n3(p):
    "n : ID"
    p[0] = p[1]

The method `p_error` is called if a syntax error occurs.  The argument `p` is the token that could not be read.  If `p` is `None` then there is a syntax error at the end of input.

In [ ]:
def p_error(_):
    pass

Setting the optional argument `write_tables` to `False` <B style="color:red">is required</B> to prevent an *obscure bug* where the parser generator tries to read an empty parse table.
We set `debug` to `True` so that the parse tables are dumped into the file `parser.out`.

In [ ]:
parser = yacc.yacc(write_tables=False, debug=True)

In [ ]:
!cat parser.out

In [ ]:
%run ../AST2Dot.ipynb

The method `test(s)` takes a string `s` that is supposed to be a `stmnt`.
This statement is then executed.

In [ ]:
def test(s):
    t = yacc.parse(s)
    d = tuple2dot(t)
    display(d)
    return t

 $\wedge$, $\vee$, $\neg$, $\top$, $\bot$

In [ ]:
test('p ∧ q')

In [ ]:
test('q ∨ p')

In [ ]:
test('¬p')

In [ ]:
test('¬p ∧ q')

In [ ]:
test('p ∧ ¬q ∨ q ∧ (¬r ∨ ¬¬p)')